In [28]:
from pathlib import Path

import pandas as pd
import numpy as np
import shutil

In [29]:
### Directories
project_root = Path.cwd()

ground_truth_directory = project_root / Path("eval/gt")
predicted_directory = project_root / Path("eval/pred")
result_directory = project_root / Path("eval/result")

ground_truth_directory.mkdir(parents=True, exist_ok=True)
predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [30]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [31]:
### Main Function
def evaluate_result(truth_csv_path, predicted_csv_path):
    
    ### Initialize Dataframes
    df_truth = pd.read_csv(truth_csv_path)
    df_pred = pd.read_csv(predicted_csv_path)


    ### Clean Up Dataframes
    df_truth.columns = df_truth.columns.str.strip()
    df_pred.columns = df_pred.columns.str.strip()


    ### Merge Both Dataframes
    merged = pd.merge(df_truth, df_pred, on=['frame_index', 'human_id', 'object_id'], how='outer', indicator=True)


    ### Checks for True Positives, False Positives, and False Negatives From The Merge DataFrame 
    tp = (merged['_merge'] == 'both').sum()
    fp = (merged['_merge'] == 'right_only').sum()
    fn = (merged['_merge'] == 'left_only').sum()
    # print(f"True Positives: {tp}")
    # print(f"False Positives: {tp}")
    # print(f"True Negatives: {tp}")


    ### Calculate Precision, Recall, and F1
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * ((precision * recall) / (precision + recall))

    return precision, recall, f1


In [32]:
# video_name = "vid17"

# gt_csv_file = ground_truth_directory / (video_name + "_true_frames.csv")
# pred_csv_file = predicted_directory / (video_name + "_pred_frames.csv")

# precision, recall, f1 = evaluate_result(gt_csv_file, pred_csv_file)

# result_text = f"{video_name:<6} Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}"
# print(result_text)

# with open(result_directory / "result.txt", "w") as result_log:
#     result_log.write(result_text)

In [33]:
results = []

for gt_csv_file in ground_truth_directory.glob("*_true_frames.csv"):

    video_name = gt_csv_file.stem[:-12]
    pred_csv_file = predicted_directory / (video_name + "_pred_frames.csv")

    precision, recall, f1 = evaluate_result(gt_csv_file, pred_csv_file)
    
    result_text = f"{video_name:<6} Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}"
    results.append(result_text)
    print(result_text)

with open(result_directory / "result.txt", "w") as result_log:
    result_log.write("\n".join(results))

vid01  Precision: 0.628 | Recall: 0.897 | F1: 0.739


In [34]:
### Main Function
def evaluate_aggregated_tiou(truth_csv_path, predicted_csv_path):

    ### Initialize Dataframes
    df_truth = pd.read_csv(truth_csv_path)
    df_pred = pd.read_csv(predicted_csv_path)

    ### Merge Both Dataframes
    df_merged = pd.merge(
        df_truth,
        df_pred,
        on=['human_id', 'object_id'],
        how='outer',
        indicator=True,
        suffixes=('_gt', '_pred')
    )
    

    ### Calculate Individual Intersections
    inter_s = np.maximum(df_merged['frame_start_gt'], df_merged['frame_start_pred'])
    inter_e = np.minimum(df_merged['frame_end_gt'], df_merged['frame_end_pred'])
    df_merged['inter'] = np.maximum(0, inter_e - inter_s + 1)


    ### Calculate Individual Lengths
    df_merged['pred_len'] = df_merged['frame_end_pred'] - df_merged['frame_start_pred'] + 1
    df_merged['gt_len'] = df_merged['frame_end_gt'] - df_merged['frame_start_gt'] + 1


    ### Format Predicted Frames String
    df_merged['frames_pred'] = df_merged['frame_start_pred'].astype(str).str.replace(r'\.0$', '', regex=True) + "->" + df_merged['frame_end_pred'].astype(str).str.replace(r'\.0$', '', regex=True)


    ### Aggregate Predictions Per Ground Truth
    df_merged = df_merged.groupby(
        ['human_id', 'object_id', 'frame_start_gt', 'frame_end_gt'], 
        as_index=False
    ).agg({
        'inter': 'sum',
        'pred_len': 'sum',
        'gt_len': 'first',
        'frames_pred': lambda x: ', '.join(x.dropna())
    })


    ### Calculate Combined Union And TIoU
    df_merged['union'] = df_merged['gt_len'] + df_merged['pred_len'] - df_merged['inter']
    df_merged['tiou'] = df_merged['inter'] / df_merged['union']
    

    ### Append distance
    df_merged["distance"] = df_truth["distance"]
    
    return df_merged

In [35]:
def convert_to_readable(df_tiou):
    df = df_tiou
    #print(df)

    df["frames_gt"] = df["frame_start_gt"].astype(str).str.replace(r'\.0$', '', regex=True).str.rjust(4) + " -> " + df["frame_end_gt"].astype(str).str.replace(r'\.0$', '', regex=True).str.ljust(4)
    df["frames_pred"] = df["frames_pred"].astype(str).apply(lambda x: ', '.join([f"{p.split('->')[0].rjust(4)} -> {p.split('->')[1].ljust(4)}" if '->' in p else p for p in x.split(', ')]))
    
    df_success = df[[
        "human_id",
        "object_id",
        "frames_gt",
        "frames_pred",
        "tiou",
        "distance"
    ]].rename(columns={
        "human_id": "Human ID",
        "object_id": "Object ID",
        "frames_gt": "Frames Truth",
        "frames_pred": "Frames Predicted",
        "tiou": "TIOU",
        "distance": "Distance"
    })

    return df_success

In [ ]:
video_name = "test"

gt_csv_file = ground_truth_directory / (video_name + "_true_summary.csv")
pred_csv_file = predicted_directory / (video_name + "_pred_summary.csv")

df_tiou = evaluate_aggregated_tiou(gt_csv_file, pred_csv_file)

df_success = convert_to_readable(df_tiou)

report_text = []
report_text.append(f"{'Human ID':>10} {'Object ID':>10} {'Frames Truth':>15} {'Frames Pred':>15} {'TIOU':>12} {'Distance':>10}")
for _, (human_id, object_id, frames_truth, frames_pred, tiou, distance) in df_success.iterrows():
    preds = str(frames_pred).split(", ")
    report_text.append(f"{human_id:>10} {object_id:>10} {frames_truth:>15} {preds[0]:>15} {tiou:>12.5f} {distance:>10}")
    for p in preds[1:]:
        report_text.append(f"{'':>10} {'':>10} {'':>15} {p:>15} {'':>12} {'':>10}")

with open(result_directory / (video_name + "_tiou.txt"), "w") as tiou_log:
    tiou_log.write("\n".join(report_text))

print("\n".join(report_text))

average_tiou = df_success["TIOU"].mean()
print(f"Average TIOU: {average_tiou}")

  Human ID  Object ID    Frames Truth     Frames Pred         TIOU   Distance
Average TIOU: nan


C:\Users\Gabriel\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in maximum
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Gabriel\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in minimum
  result = getattr(ufunc, method)(*inputs, **kwargs)


: 